In [2]:
import pandas as pd
df = pd.read_csv('./df.csv')

print(df.head())

        venue              address        tel        lat        long  \
0      香港仔體育館  香港仔大道203號香港仔市政大廈5字樓  2555 8909  22.249317  114.154483   
1  香港仔網球及壁球中心             香港仔海傍道1號  2553 6130  22.247500  114.159959   
2      鴨脷洲體育館             鴨脷洲洪聖街8號  2554 0832  22.244516  114.155361   
3       柴灣體育館                柴灣怡順街  2897 9144  22.264477  114.239901   
4      港灣道體育館             灣仔港灣道27號  2827 9684  22.281276  114.176853   

                                             href    id   area  滑雪機  划艇機  跑步機  
0  /clpss/tc/webApp/FitnessRoomDetails.do?id=1374  1374  165.0    2    2    4  
1  /clpss/tc/webApp/FitnessRoomDetails.do?id=1375  1375  125.0    2    2    4  
2  /clpss/tc/webApp/FitnessRoomDetails.do?id=1376  1376  115.0    0    1    4  
3  /clpss/tc/webApp/FitnessRoomDetails.do?id=1365  1365  145.0    2    2    5  
4  /clpss/tc/webApp/FitnessRoomDetails.do?id=1440  1440  254.0    2    2    0  


# Unzip and read shp file 

In [3]:
import zipfile
import geopandas as gpd

df_shapes = gpd.read_file('./DCCA_2019_Shapefile/DCCA_2019.shp', encoding='big5')

In [4]:
df_shapes

,CACODE,ENAME,CNAME,Shape_Leng,Shape_Area,geometry
0,B11,Stubbs Road,司徒拔道,12833.520088,4.203165e+06,"POLYGON ((836535.728 815040.463, 836534.252 81..."
1,B10,Happy Valley,跑馬地,3606.402113,4.450347e+05,"POLYGON ((837180.285 814553.135, 837192.132 81..."
2,B09,Broadwood,樂活,3625.862186,5.284740e+05,"POLYGON ((836887.966 815389.851, 836909.444 81..."
3,B08,Jardine's Lookout,渣甸山,9306.961507,2.193531e+06,"POLYGON ((838164.914 815238.444, 838166.600 81..."
4,B07,Tai Hang,大坑,4120.098581,4.049233e+05,"POLYGON ((837847.564 815747.244, 837853.000 81..."
...,...,...,...,...,...,...
447,Q02,Pak Sha Wan,白沙灣,24550.341462,2.311459e+07,"POLYGON ((844628.630 827187.132, 844630.438 82..."
448,Q03,Sai Kung Islands,西貢離島,85557.880727,3.306551e+08,"POLYGON ((865002.174 836329.660, 869803.000 82..."
449,Q11,Po Yee,寶怡,1249.359141,9.041565e+04,"POLYGON ((845221.766 818830.987, 845272.485 81..."
450,Q20,Wai Yan,慧茵,1349.204367,6.713569e+04,"POLYGON ((844485.079 820505.154, 844487.381 82..."


# Check whether the point is in the shape

In [5]:
def latlong2HK1980(lat, lng):
    import urllib.request, urllib.parse, urllib.error
    import json

    serviceurl = 'http://www.geodetic.gov.hk/transform/v2/'

    parms = dict()
    parms['inSys'] = 'wgsgeog'
    parms['outSys'] = 'hkgrid'
    parms['lat'] = lat
    parms['long'] = lng
    url = serviceurl + '?' + urllib.parse.urlencode(parms)

    uh = urllib.request.urlopen(url)
    data = uh.read().decode()
    js = json.loads(data)

    hkN = js['hkN']
    hkE = js['hkE']

    return (hkE, hkN)  

In [8]:
from shapely.geometry import Point, Polygon

df_dist = pd.DataFrame(columns=['venue', 'dist_code', 'dist_name'])

for _, row in df.iterrows():
    p = Point(latlong2HK1980(row.lat, row.long))
    for _, shapes_row in df_shapes.iterrows():
        if p.within(shapes_row.geometry):
            dist_row = {'venue':row.venue, 'dist_code':shapes_row.CACODE, 'dist_name':shapes_row.CNAME}
            print(dist_row)
            df_dist = df_dist.append(dist_row, ignore_index=True)

{'venue': '香港仔體育館', 'dist_code': 'D01', 'dist_name': '香港仔'}
{'venue': '香港仔網球及壁球中心', 'dist_code': 'D15', 'dist_name': '黃竹坑'}
{'venue': '鴨脷洲體育館', 'dist_code': 'D03', 'dist_name': '鴨?洲北'}
{'venue': '柴灣體育館', 'dist_code': 'C33', 'dist_name': '翠德'}
{'venue': '港灣道體育館', 'dist_code': 'B01', 'dist_name': '軒尼詩'}
{'venue': '香港公園體育館', 'dist_code': 'A01', 'dist_name': '中環'}
{'venue': '港島東體育館', 'dist_code': 'C03', 'dist_name': '鯉景灣'}
{'venue': '駱克道體育館', 'dist_code': 'B01', 'dist_name': '軒尼詩'}
{'venue': '鰂魚涌體育館', 'dist_code': 'C15', 'dist_name': '柏架山'}
{'venue': '石塘咀體育館', 'dist_code': 'A15', 'dist_name': '水街'}
{'venue': '上環體育館', 'dist_code': 'A12', 'dist_name': '上環'}
{'venue': '士美非路體育館', 'dist_code': 'A08', 'dist_name': '西環'}
{'venue': '中山紀念公園體育館', 'dist_code': 'A11', 'dist_name': '西營盤'}
{'venue': '彩虹道體育館', 'dist_code': 'H02', 'dist_name': '龍下'}
{'venue': '竹園體育館', 'dist_code': 'H13', 'dist_name': '翠竹及鵬程'}
{'venue': '花園街體育館', 'dist_code': 'E15', 'dist_name': '旺角南'}
{'venue': '佛光街體育館', 'dist_code': 'G24

In [18]:
import string

letter2dist = {'A': '中西區', 'B': '灣仔區', 'C':'東區', 'D':'南區'
               , 'E': '油尖旺區', 'F': '深水埗區', 'G':'九龍城區', 'H':'黃大仙區', 'J':'觀塘區'
               , 'K':'荃灣區', 'L':'屯門區', 'M':'元朗區'
               , 'N':'北區', 'P':'大埔區', 'Q':'西貢區', 'R':'沙田區'
               , 'S':'葵青區', 'T':'離島區'
              }
letter2region = {}
for letter in string.ascii_uppercase:
    if letter in 'ABCD':
        letter2region[letter] = '港島'
    if letter in 'EFG':
        letter2region[letter] = '九龍西'
    if letter in 'HJ':
        letter2region[letter] = '九龍東'
    if letter in 'KLMST':
        letter2region[letter] = '新界西'
    if letter in 'NPQR':
        letter2region[letter] = '新界東'

In [20]:
df_dist['district'] = df_dist.dist_code.str[:1].map(letter2dist)
df_dist['region'] = df_dist.dist_code.str[:1].map(letter2region)

In [21]:
df_dist.to_csv('df_dist.csv')

,venue,dist_code,dist_name,district,region
0,香港仔體育館,D01,香港仔,南區,港島
1,香港仔網球及壁球中心,D15,黃竹坑,南區,港島
2,鴨脷洲體育館,D03,鴨?洲北,南區,港島
3,柴灣體育館,C33,翠德,東區,港島
4,港灣道體育館,B01,軒尼詩,灣仔區,港島
...,...,...,...,...,...
75,友愛體育館,L06,友愛北,屯門區,新界西
76,楊屋道體育館,K03,荃灣南,荃灣區,新界西
77,圓洲角體育館,R37,愉欣,沙田區,新界東
78,元朗體育館,M01,豐年,元朗區,新界西
